# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library. We'll walk through metadata loading, overviewing record sets and fields, extracting data using entity `@id`s, and performing basic exploratory data analysis.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and show their fields
print("Record sets in dataset:")
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()
# Save the first recordset's @id for examples below
record_set_id = metadata.record_sets[0].id
# Also list all available record set ids for later use
all_record_set_ids = [rs.id for rs in metadata.record_sets]
# Optionally, print available field ids for the first record set
field_ids = [field.id for field in metadata.record_sets[0].fields]
print(f"First record set id: {record_set_id}")print(f"Field ids in first record set: {field_ids}")

## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` for analysis. All references (record set, field) use their Croissant `@id`s.

In [ ]:
# Extract data from each record set using its @id
record_sets = all_record_set_ids
dataframes = {}

for record_set in record_sets:
    # Each record is a dict mapping field @id to value
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

example_rs = record_set_id  # Use the first record set for demonstration
print(f"Columns in {example_rs} DataFrame:")
print(dataframes[example_rs].columns.tolist())
dataframes[example_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id`
- Filtering records by a numeric field
- Normalizing data
- Grouping by a categorical field

Note: Refer to earlier output for available field `@id`s for selection below.

In [ ]:
# Example: Filter and analyze using known numeric and group fields identified above
# Please update <numeric_field_id> and <group_field_id> below for your actual dataset fields

# For demonstration, select the first numeric field id (if any)
import numpy as np
df = dataframes[example_rs]

# Identify a numeric field by checking dtype
numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    # If not detected automatically, set manually (replace with actual field @id as needed)
    numeric_field = field_ids[0]

# Select a group field (categorical)
non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
group_field = non_numeric_fields[0] if non_numeric_fields else None

print(f"Using numeric field @id: {numeric_field}")
if group_field:
    print(f"Using group field @id: {group_field}")
else:
    print("No groupable categorical field detected.")

threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group (if available)
if group_field and group_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f'{numeric_field} per {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² mass spectrometry dataset with `mlcroissant`. You viewed record sets and fields by `@id`, extracted the data, and performed filtering, normalization, grouping, and visualization for initial analysis. For more targeted research, refine the selected field `@id`s based on the domain context and Croissant schema documentation.